# Hafta 11 — Modern LLM Uygulamaları: RAG, Araçlar, Değerlendirme

Modelin ötesi: LLM'leri pratikte kullanışlı kılan sistemler ve değerlendirme çerçeveleri. Sıfırdan bir erişimle zenginleştirilmiş üretim hattı kuruyor, yapılandırılmış araç çağırmayı uyguluyor ve modern kıyaslamaların yöntemsel tuzaklarını tartışıyoruz.

## Öğrenme Hedefleri

- Parçalamadan üretime kadar, hem yoğun hem seyrek erişim içeren bir erişimle zenginleştirilmiş üretim (RAG) hattı kurmak.
- Asgari bir ajan döngüsüyle yapılandırılmış araç çağırmayı uygulamak.
- Değerlendirme düzeneklerini doğru kullanmak: güven aralıklarını raporlamak, veri kirlenmesini tespit etmek, görev tabanlı değerlendirmeyi model tarafından puanlanan değerlendirmeden ayırt etmek.

## Zorunlu Okumalar

- Lewis, P. vd. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*.
- Karpukhin, V. vd. (2020). *Dense Passage Retrieval for Open-Domain Question Answering*.
- Liang, P. vd. (2022). *Holistic Evaluation of Language Models*.

In [ ]:
import sys, re, math, json, random
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
np.random.seed(0); random.seed(0)

## 1. Neden erişimle zenginleştirilmiş üretim?

Parametrik bir dil modeli bilgiyi ağırlıklarında saklar — şunları yapamaz:

- belirli bir kaynağa atıf yapmak,
- yeniden eğitilmeden güncellenmek,
- özel veya kuruma ait derlemlerle çalışmak.

**RAG** (Lewis vd., 2020) bunları, çıkarım anında ilgili belgeleri getirip üretimi onlara koşullayarak aşar. Mimarinin dört aşaması vardır:

> **parçalama → gömme → erişim → üretim**

Her birini sıfırdan kuruyoruz.

## 2. Oyuncak bir bilgi tabanı

Erişimin önemsiz olmayacağı kadar tekrar ve örtüşme içeren minyatür bir ansiklopedi (İngilizce metinler).

In [ ]:
KB = [
    "The Eiffel Tower is a wrought-iron lattice tower in Paris, France. It was completed in 1889.",
    "Gustave Eiffel's company designed the Eiffel Tower for the 1889 World's Fair in Paris.",
    "The Statue of Liberty is a copper statue in New York Harbor, gifted by France in 1886.",
    "Frédéric Auguste Bartholdi designed the Statue of Liberty; Gustave Eiffel engineered its internal frame.",
    "The Great Wall of China stretches over 21,000 kilometers across northern China.",
    "Construction of the Great Wall began as early as the 7th century BC under various dynasties.",
    "The Colosseum is an oval amphitheatre in the centre of Rome, completed in 80 AD.",
    "The Roman Colosseum could hold an estimated 50,000 to 80,000 spectators during gladiatorial contests.",
    "Mount Everest is the Earth's highest mountain above sea level, located in the Himalayas.",
    "Mount Everest's official elevation, measured most recently in 2020, is 8,848.86 metres.",
    "Machu Picchu is a 15th-century Inca citadel located on a mountain ridge in southern Peru.",
    "Machu Picchu was built around 1450 and abandoned a century later during the Spanish conquest.",
]
print(f"Bilgi tabanında {len(KB)} parça var.")

## 3. Seyrek erişim — BM25

BM25 (Robertson ve Walker, 1994), klasik bilgi erişiminin beygir gücüdür:

$$\text{BM25}(q, d) = \sum_{t \in q} \text{idf}(t) \cdot \frac{f(t, d) (k_1 + 1)}{f(t, d) + k_1 (1 - b + b \cdot |d| / \bar{|d|})}.$$

$k_1 \approx 1.5$ ve $b \approx 0.75$ ile BM25, anahtar sözcük ağırlıklı sorgular için yenilmesi zor bir taban çizgisidir — ve en önemlisi, *hiç öğrenilebilir parametresi yoktur*.

In [ ]:
class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b

    def fit(self, docs):
        self.docs_tokens = [re.findall(r"\w+", d.lower()) for d in docs]
        self.N = len(docs)
        self.avgdl = np.mean([len(d) for d in self.docs_tokens])
        # df[t] = t'yi içeren belge sayısı
        self.df = Counter()
        for d in self.docs_tokens:
            for t in set(d):
                self.df[t] += 1
        # Robertson-Spärck-Jones idf, alttan 0 ile sınırlı.
        self.idf = {t: max(0.0, math.log((self.N - df + 0.5) / (df + 0.5)))
                    for t, df in self.df.items()}
        # Belge başına terim sıklıklarını önceden hesapla.
        self.tf = [Counter(d) for d in self.docs_tokens]
        return self

    def score(self, query, doc_idx):
        q_tokens = re.findall(r"\w+", query.lower())
        d = self.docs_tokens[doc_idx]; tf = self.tf[doc_idx]
        L = len(d)
        s = 0.0
        for t in q_tokens:
            if t not in self.idf:
                continue
            f = tf.get(t, 0)
            denom = f + self.k1 * (1 - self.b + self.b * L / self.avgdl)
            s += self.idf[t] * (f * (self.k1 + 1)) / max(denom, 1e-9)
        return s

    def search(self, query, k=3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x: -x[1])[:k]

bm25 = BM25().fit(KB)
for q in ["Who designed the Eiffel Tower?", "How tall is Mount Everest?",
          "When was Machu Picchu built?"]:
    hits = bm25.search(q, k=2)
    print(f"\nS: {q}")
    for i, s in hits:
        print(f"  [{s:5.2f}]  {KB[i]}")

## 4. Yoğun erişim — gömme tabanlı

Sözcüksel eşleşmeyi bir gömme uzayındaki kosinüs benzerliğiyle değiştirin. Üretimde bu bir sentence-transformer modeli olurdu. Burada not defterini kendi kendine yeterli tutmak için deterministik, karma (hash) tabanlı bir öznitelik torbası kodlayıcısı kullanıyoruz — güçlü bir erişimci *değil*, ama mekanik olarak gerçeğiyle aynı.

Projenizde kodlayıcıyı `sentence-transformers/all-MiniLM-L6-v2` ile değiştirin; kodun geri kalanı değişmeden kalır.

In [ ]:
class HashedEncoder:
    """Oyuncak kodlayıcı: sözcük torbasını sabit boyutlu bir vektöre karmalar.
    Pratikte gerçek bir cümle kodlayıcısıyla değiştirin."""
    def __init__(self, dim=128, seed=0):
        self.dim = dim
        self.rng = np.random.default_rng(seed)
        self.hashes = {}

    def _hash(self, t):
        if t not in self.hashes:
            self.hashes[t] = self.rng.normal(size=self.dim)
        return self.hashes[t]

    def encode(self, text):
        tokens = re.findall(r"\w+", text.lower())
        if not tokens:
            return np.zeros(self.dim)
        v = np.sum([self._hash(t) for t in tokens], axis=0)
        n = np.linalg.norm(v)
        return v / n if n > 0 else v

enc = HashedEncoder(dim=256)
doc_vecs = np.stack([enc.encode(d) for d in KB])

def dense_search(query, k=3):
    q = enc.encode(query)
    sims = doc_vecs @ q
    idx = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i])) for i in idx]

print("Yoğun erişim:")
for q in ["Who designed the Eiffel Tower?", "How tall is Mount Everest?"]:
    print(f"\nS: {q}")
    for i, s in dense_search(q, k=2):
        print(f"  [{s:.3f}]  {KB[i]}")

## 5. Karma (hibrit) erişim

Yoğun ve seyrek erişim birbirini tamamlayan hatalar yapar. Üretim sistemleri bunları genellikle karşılıklı sıra birleştirmesiyle (reciprocal rank fusion, RRF) birleştirir:

$$\text{RRF}(d) = \sum_{r \in \text{erişimciler}} \frac{1}{k + \text{sıra}_r(d)},$$

burada sıra 1'den başlar (en iyi belgenin sırası 1'dir) ve $k = 60$ yaygın varsayılandır.

In [ ]:
def rrf(rankings, k=60, top_k=3):
    scores = Counter()
    for ranked in rankings:
        for rank, (doc_id, _) in enumerate(ranked, start=1):   # sıra 1'den başlar
            scores[doc_id] += 1.0 / (k + rank)
    return scores.most_common(top_k)

q = "Who built the iron lattice tower in Paris?"
sparse_hits = bm25.search(q, k=5)
dense_hits = dense_search(q, k=5)
fused = rrf([sparse_hits, dense_hits], top_k=3)
print(f"S: {q}\n")
print("Seyrek ilk 3:", [i for i, _ in sparse_hits[:3]])
print("Yoğun  ilk 3:", [i for i, _ in dense_hits[:3]])
print("Birleşik ilk 3:", [i for i, _ in fused])
for i, _ in fused:
    print(f"  → {KB[i]}")

## 6. Hepsini bir araya getirmek: bir RAG istem şablonu

Erişim sonuçları bir istem şablonuna yerleştirilip bir üreticiye verilir. Gerçek bir LLM API'si veya yerel bir modelle bu tek bir çağrıdır. Burada yalnızca istemin kurulmasını gösteriyoruz. (Bağlam belgeleri İngilizce olsa da talimat Türkçedir; modern LLM'ler yanıtı talimatın dilinde üretebilir.)

In [ ]:
def rag_prompt(query, retrieved_docs):
    context = "\n".join(f"[{i+1}] {d}" for i, d in enumerate(retrieved_docs))
    return (
        "Soruyu yalnızca aşağıdaki bağlamı kullanarak yanıtla. "
        "Bağlam yanıtı içermiyorsa bunu açıkça söyle. "
        "Kaynakları köşeli parantez içindeki numaralarıyla göster.\n\n"
        f"Bağlam:\n{context}\n\n"
        f"Soru: {query}\n"
        "Yanıt:"
    )

q = "Who engineered the internal frame of the Statue of Liberty?"
hits = [i for i, _ in rrf([bm25.search(q, 5), dense_search(q, 5)], top_k=3)]
prompt = rag_prompt(q, [KB[i] for i in hits])
print(prompt)

## 7. Erişimi değerlendirmek

Standart metrikler:

- **Recall@k** — doğru pasajın ilk $k$ sonuç arasında bulunduğu sorguların oranı.
- **MRR (ortalama karşılıklı sıra)** — $\frac{1}{|Q|} \sum_q \frac{1}{\text{sıra}_q}$; burada $\text{sıra}_q$, ilk doğru pasajın konumudur.
- **nDCG@k** — konuma göre indirimli, dereceli ilgililik.

Küçük, etiketlenmiş bir küme üzerinde recall@k'yi hesaplıyoruz.

In [ ]:
# (sorgu, bilgi tabanındaki doğru belgenin indeksi)
QUERIES = [
    ("Who designed the Eiffel Tower?", 1),
    ("When was the Eiffel Tower completed?", 0),
    ("Who engineered the Statue of Liberty's frame?", 3),
    ("How long is the Great Wall?", 4),
    ("When did Great Wall construction begin?", 5),
    ("How many spectators did the Colosseum hold?", 7),
    ("What is Mount Everest's elevation?", 9),
    ("When was Machu Picchu built?", 11),
]

def recall_at_k(retriever_fn, queries, k=3):
    hits = 0
    for q, gold in queries:
        results = retriever_fn(q, k=k)
        if gold in [r[0] for r in results]:
            hits += 1
    return hits / len(queries)

def hybrid_search(q, k=3):
    return rrf([bm25.search(q, 5), dense_search(q, 5)], top_k=k)

print(f"{'Erişimci':<10} {'R@1':>6} {'R@3':>6} {'R@5':>6}")
for name, fn in [('BM25', bm25.search), ('Yoğun', dense_search), ('Karma', hybrid_search)]:
    r1 = recall_at_k(fn, QUERIES, 1)
    r3 = recall_at_k(fn, QUERIES, 3)
    r5 = recall_at_k(fn, QUERIES, 5)
    print(f"{name:<10} {r1:>6.2f} {r3:>6.2f} {r5:>6.2f}")

## 8. Yapılandırılmış araç çağırma

Modern LLM'lere bir aracı ve argümanlarını adlandıran yapılandırılmış JSON üretmeleri istenebilir. Basit bir ajan döngüsü:

```
while not done:
    response = LLM(history)
    if response contains tool_call:
        result = execute_tool(tool_call)
        history += result
    else:
        return response
```

Döngü tamamen çalıştırılabilsin diye LLM'i düzenli ifade tabanlı bir taslakla taklit ediyoruz. Döngünün yapısı gerçek bir modelle de aynıdır — yalnızca `fake_llm` fonksiyonu değişir.

> **Güvenlik notu.** Araç argümanları modelden gelir, yani güvenilmez girdidir. Hesap makinesinde `eval(expr, {"__builtins__": {}})` kullanmak güvenli **değildir**: `().__class__.__base__.__subclasses__()` gibi ifadelerle Python'un iç sınıflarına erişilebilir. Aşağıdaki hesap makinesi ifadeyi `ast` ile ayrıştırır ve yalnızca sayıları ve aritmetik işleçleri değerlendirir.

In [ ]:
import ast, operator

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod,
        ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def safe_calculator(expr):
    '''Yalnızca sayıları ve aritmetik işleçleri değerlendirir; başka her şeyi reddeder.'''
    def ev(node):
        if isinstance(node, ast.Expression):
            return ev(node.body)
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            left, right = ev(node.left), ev(node.right)
            if isinstance(node.op, ast.Pow) and abs(right) > 100:
                raise ValueError("üs çok büyük")
            return _OPS[type(node.op)](left, right)
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError(f"izin verilmeyen ifade öğesi: {type(node).__name__}")
    return ev(ast.parse(expr, mode="eval"))

TOOLS = {
    # Arama aracı, LLM'e geri verilecek belge METİNLERİNİ döndürür (yalnızca kimlik/skor değil).
    "search": lambda q: [KB[i] for i, _ in hybrid_search(q, k=2)],
    "kb_get": lambda i: KB[int(i)],
    "calculator": lambda expr: safe_calculator(expr),
}

def fake_llm(history):
    '''Küçük senaryolu bir politika: olgusal sorular için arama çağrısı, kullanıcı
    "compute" derse hesap makinesi çağrısı, aksi hâlde nihai yanıt üretir.'''
    last_user = next(m['content'] for m in reversed(history) if m['role'] == 'user')
    last_tool = next((m for m in reversed(history) if m['role'] == 'tool'), None)
    if 'compute' in last_user.lower() and last_tool is None:
        expr = re.search(r"compute\s+(.+)", last_user, re.IGNORECASE)
        return {"tool": "calculator", "args": {"expr": expr.group(1).strip().rstrip("?.")}}
    if last_tool is None:
        return {"tool": "search", "args": {"q": last_user}}
    content = last_tool['content']
    evidence = content[0] if isinstance(content, list) else content
    return {"final": f"({last_tool['name']} sonucuna dayanan yanıt): {str(evidence)[:200]}"}

def run_agent(user_query, max_steps=4):
    history = [{"role": "user", "content": user_query}]
    for step in range(max_steps):
        decision = fake_llm(history)
        if "final" in decision:
            print(f"  [nihai] {decision['final']}")
            return decision['final']
        tool, args = decision['tool'], decision['args']
        print(f"  [adım {step}] {tool}({args}) çağrılıyor")
        try:
            result = TOOLS[tool](**args)
        except Exception as e:  # araç hatası LLM'e gözlem olarak geri verilir
            result = f"HATA: {e}"
        history.append({"role": "assistant", "content": json.dumps(decision)})
        history.append({"role": "tool", "name": tool, "content": result})
    return "(en fazla adım sayısına ulaşıldı)"

for q in ["Who designed the Eiffel Tower?", "compute 2 ** 10 - 24",
          "compute ().__class__.__base__.__subclasses__()"]:
    print(f"\nS: {q}")
    run_agent(q)

## 9. Değerlendirme tuzakları

LLM kıyaslama sayılarını raporlarken eksiksiz olmayan bir denetim listesi:

1. **Güven aralıkları.** Çoğu açık kıyaslama varyanssız tek bir sayı raporlar. Test kümesini bootstrap ile yeniden örnekleyin veya çok tohumlu değerlendirme kullanın. Liang vd. (2022), güven aralıkları örtüşen küçük modellerin rutin olarak "X, Y'den daha iyi" diye raporlandığını buldu.
2. **Kirlenme.** Test kümeleri ön eğitim derlemlerine sızar. Basit bir kontrol: test örneklerinin uzun n-gramları ön eğitim derleminde birebir geçiyor mu? (GPT-3 makalesi 13-gram örtüşmesi kullanır.) Daha güçlü bir sonda: dil modeli gizlenmiş bir kıyaslama örneğini ne sıklıkla birebir tamamlıyor? Oran yüksekse sonuçlar şüphelidir (Magar ve Schwartz, 2022).
3. **İstem duyarlılığı.** Farklı istem biçimleri puanları onlarca puan değiştirebilir. Kullanılan istem şablonunu raporlayın ve birkaç varyantın ortalamasını almayı düşünün.
4. **Model tarafından puanlanan değerlendirmeler.** Daha güçlü bir LLM'i hakem olarak kullanmak (LLM-as-judge) hızlıdır ama hakemin kendi tercihlerine doğru yanlılık yaratır (Zheng vd., 2023). Bir örneklem üzerinde insan değerlendirmesiyle çapraz kontrol edin.

İlk ikisini deneysel olarak gösteriyoruz.

In [ ]:
# 1) Küçük bir değerlendirme kümesinde doğruluk için bootstrap güven aralığı.
def bootstrap_ci(correct_flags, n_iter=2000, alpha=0.05):
    correct_flags = np.asarray(correct_flags)
    n = len(correct_flags)
    stats = []
    for _ in range(n_iter):
        idx = np.random.randint(0, n, size=n)
        stats.append(correct_flags[idx].mean())
    lo, hi = np.percentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return correct_flags.mean(), lo, hi

# İki sistemin de 20 maddelik bir değerlendirmede %80 aldığını varsayalım.
A = [1]*16 + [0]*4
B = [1]*16 + [0]*4
random.shuffle(A); random.shuffle(B)
acc_A, lo_A, hi_A = bootstrap_ci(A)
acc_B, lo_B, hi_B = bootstrap_ci(B)
print(f"Sistem A: {acc_A:.2f}  %95 GA [{lo_A:.2f}, {hi_A:.2f}]")
print(f"Sistem B: {acc_B:.2f}  %95 GA [{lo_B:.2f}, {hi_B:.2f}]")
print("n=20 iken güven aralığı kabaca ±0.18'dir — burada 'A, B'yi 5 puan geçiyor' demek gürültüdür.")

# 2) Kirlenme denetimi: test sorularının n-gramları "ön eğitim derleminde" birebir geçiyor mu?
def ngrams_of(text, n):
    toks = re.findall(r"\w+", text.lower())
    return {tuple(toks[i:i + n]) for i in range(len(toks) - n + 1)}

def contamination_rate(test_items, corpus_docs, n=8):
    corpus_grams = set().union(*(ngrams_of(d, n) for d in corpus_docs))
    flagged = [q for q in test_items if ngrams_of(q, n) & corpus_grams]
    return len(flagged) / len(test_items), flagged

# Oyuncak bir "ön eğitim derlemi": bilgi tabanı + yanlışlıkla sızmış bir test sorusu.
pretraining_corpus = KB + ["forum post: q: how many spectators did the colosseum hold during gladiatorial contests? a: 50,000 to 80,000"]
test_questions = [
    "How many spectators did the Colosseum hold during gladiatorial contests?",   # sızmış
    "Which company designed the tower completed for the 1889 fair?",              # temiz
    "What is the height of the tallest mountain on Earth?",                       # temiz
]
rate, flagged = contamination_rate(test_questions, pretraining_corpus, n=8)
print(f"\n8-gram kirlenme oranı: {rate:.2f}")
for q in flagged:
    print(f"  kirlenmiş olarak işaretlendi: {q!r}")

## 10. Alıştırmalar

1. **Gerçek bir RAG hattı.** `HashedEncoder`'ı `sentence-transformers/all-MiniLM-L6-v2` ile değiştirin, 1000'den fazla belgeyi FAISS ile indeksleyin ve küçük bir soru-yanıt kümesinde uçtan uca yanıt kalitesini (birebir eşleşme) ölçün.
2. **Parçalama ablasyonu.** $\{64, 128, 256, 512\}$ token parça boyutlarını ve $\{0, 25, 50\}$% örtüşmeyi deneyin. Erişim recall'unu parça boyutuna göre çizdirin. En uygun nokta nerede?
3. **Araç çağırma sağlamlığı.** Ajanın, art arda *birden çok* araç çağrısı gerektiren soruları yanıtlamasını sağlayın (ör. "Everest'in yüksekliğinin ortalama gladyatör seyirci sayısına oranını hesapla"). Ajanın ne sıklıkla doğru yaptığını ve hangi hata türlerinin ortaya çıktığını izleyin.
4. **Kirlenme sondası.** Herhangi bir açık kıyaslamadan beş örnek seçin. Herkese açık bir LLM'e her birinin ilk yarısını verin ve ikinci yarısını ne sıklıkla birebir tamamladığına bakın. Oranı raporlayın. Yukarıdaki n-gram denetiminde $n$'yi değiştirmenin yanlış pozitif / yanlış negatif oranlarını nasıl etkilediğini tartışın.

---

## Gelecek Hafta

Hafta 12 — Bitirme projesi. Uçtan uca alana özgü dil modeli projesi.